# Python Classes and Objects — Interview Preparation

Covers: class definition, __init__, instance vs class variables, methods, __repr__/__str__, @property, and interview Q&A.

## 1. Class Definition and __init__

In [ ]:
class Dog:
    # Class variable — shared by ALL instances
    species = 'Canis familiaris'
    count = 0

    def __init__(self, name, age, breed):
        # Instance variables — unique to each instance
        self.name = name
        self.age = age
        self.breed = breed
        Dog.count += 1

    def __repr__(self):
        return f'Dog(name={self.name!r}, age={self.age}, breed={self.breed!r})'

    def __str__(self):
        return f'{self.name} ({self.breed}, {self.age} years old)'

    def bark(self):
        return f'{self.name} says: Woof!'


rex = Dog('Rex', 3, 'German Shepherd')
buddy = Dog('Buddy', 5, 'Golden Retriever')

print(repr(rex))   # uses __repr__
print(str(rex))    # uses __str__
print(rex)         # uses __str__
print([rex])       # uses __repr__ in containers
print('Dog count:', Dog.count)

## 2. Instance vs Class Variables — The Mutable Default Pitfall

In [ ]:
# PITFALL: Mutable class variable shared across instances
class BadClass:
    items = []  # shared!

    def add(self, item):
        self.items.append(item)

b1 = BadClass()
b2 = BadClass()
b1.add('x')
print('b2.items after b1.add:', b2.items)  # ['x'] — b2 is affected!

# FIX: Initialize mutable defaults in __init__
class GoodClass:
    def __init__(self):
        self.items = []  # each instance gets its own list

    def add(self, item):
        self.items.append(item)

g1 = GoodClass()
g2 = GoodClass()
g1.add('x')
print('g2.items after g1.add:', g2.items)  # [] — unaffected

## 3. Instance, Class, and Static Methods

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    # Instance method — has access to self
    def to_fahrenheit(self):
        return self._celsius * 9/5 + 32

    def __repr__(self):
        return f'Temperature({self._celsius}°C)'

    # Class method — alternative constructor pattern
    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        return cls((fahrenheit - 32) * 5/9)

    @classmethod
    def from_kelvin(cls, kelvin):
        return cls(kelvin - 273.15)

    # Static method — utility, no cls or self
    @staticmethod
    def is_valid_celsius(value):
        return value >= -273.15


t1 = Temperature(100)
print('100°C in Fahrenheit:', t1.to_fahrenheit())

t2 = Temperature.from_fahrenheit(212)  # alternative constructor
print('212°F:', t2)

t3 = Temperature.from_kelvin(373.15)
print('373.15K:', t3)

print('Is -300 valid?', Temperature.is_valid_celsius(-300))
print('Is 25 valid?', Temperature.is_valid_celsius(25))

## 4. @property Decorator

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius  # _ prefix = private by convention

    @property
    def radius(self):
        """Getter — accessed like an attribute"""
        return self._radius

    @radius.setter
    def radius(self, value):
        """Setter — validates before storing"""
        if value < 0:
            raise ValueError(f'Radius cannot be negative: {value}')
        self._radius = value

    @property
    def area(self):
        """Computed property — no setter"""
        import math
        return math.pi * self._radius ** 2

    @property
    def diameter(self):
        return self._radius * 2

    @diameter.setter
    def diameter(self, value):
        self._radius = value / 2  # reuses radius setter logic


c = Circle(5)
print('radius:', c.radius)    # calls getter
print('area:', c.area)        # computed property
print('diameter:', c.diameter)

c.radius = 10                 # calls setter
print('After setting radius=10, area:', c.area)

c.diameter = 20               # calls diameter setter
print('After setting diameter=20, radius:', c.radius)

try:
    c.radius = -1             # raises ValueError
except ValueError as e:
    print('Error:', e)

## 5. __slots__

In [ ]:
import sys

class WithDict:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlots:
    __slots__ = ['x', 'y']  # no __dict__, fixed attributes
    def __init__(self, x, y):
        self.x = x
        self.y = y

d = WithDict(1, 2)
s = WithSlots(1, 2)

print(f'WithDict size: {sys.getsizeof(d)} bytes, has __dict__: {hasattr(d, "__dict__")}')
print(f'WithSlots size: {sys.getsizeof(s)} bytes, has __dict__: {hasattr(s, "__dict__")}')

# Can add arbitrary attributes to WithDict
d.z = 3
print('d.z:', d.z)

# Cannot add arbitrary attributes to WithSlots
try:
    s.z = 3
except AttributeError as e:
    print('Slots error:', e)

## 6. Interview Practice

In [ ]:
# Q: type() vs isinstance()
class Animal: pass
class Dog(Animal): pass

d = Dog()
print('type(d) is Dog:', type(d) is Dog)       # True
print('type(d) is Animal:', type(d) is Animal) # False — exact type only
print('isinstance(d, Dog):', isinstance(d, Dog))     # True
print('isinstance(d, Animal):', isinstance(d, Animal))  # True — respects inheritance

# isinstance with tuple of types
print('isinstance(d, (Dog, str)):', isinstance(d, (Dog, str)))  # True

In [ ]:
# Q: Implement a BankAccount class with validation
class BankAccount:
    interest_rate = 0.02  # class variable

    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance
        self._transactions = []

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('Deposit must be positive')
        self._balance += amount
        self._transactions.append(('deposit', amount))

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('Withdrawal must be positive')
        if amount > self._balance:
            raise ValueError('Insufficient funds')
        self._balance -= amount
        self._transactions.append(('withdraw', amount))

    @classmethod
    def set_interest_rate(cls, rate):
        cls.interest_rate = rate

    def __repr__(self):
        return f'BankAccount(owner={self.owner!r}, balance={self._balance})'


acc = BankAccount('Alice', 1000)
acc.deposit(500)
acc.withdraw(200)
print(acc)
print('Balance:', acc.balance)
print('Transactions:', acc._transactions)